# Training

### Setup

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Setup working directory
from pathlib import Path
import os

def find_root_dir(marker='tfg'):
    p = Path.cwd()
    for candidate in [p] + list(p.parents):
        if candidate.name == marker:
            return candidate.resolve()
        if (candidate / marker).is_dir():
            return (candidate / marker).resolve()
    raise FileNotFoundError(f"Could not find '{marker}' folder in {Path.cwd()} or its parents.")

os.chdir(find_root_dir('tfg'))

## Load Datasets and Models

In [3]:
DATASETS = {
    "correlation_098": "data/processed/stage2/full-225k-correlation_098.parquet",
    "filtered": "data/processed/stage2/full-225k-filtered.parquet",
    "log": "data/processed/stage2/full-225k-log.parquet",
}

In [4]:
from src.models import get_full_dataset_models
models = get_full_dataset_models()

In [5]:
from src.preprocessing import load_dataset, split_features_target, encode_target

X_SETS = {}
Y_SETS = {}

for feature_set_name, path in DATASETS.items():

    df = load_dataset(path)
    X, y = split_features_target(df,target_column="Label")
    y = encode_target(y)

    X_SETS[feature_set_name] = X
    Y_SETS[feature_set_name] = y

In [6]:
for name, X in X_SETS.items():
    print(f"{name}: {X.shape}")

correlation_098: (225603, 43)
filtered: (225603, 59)
log: (225603, 59)


## Train

In [7]:
from src.train import train
saved = train(
    X_SETS,
    Y_SETS,
    models,
    dir_name="models/stage2/unscaled",
    scaler=None,
    test_size=0.1
)

Training models on every feature set

========== Logistic Regression (models\stage2\unscaled\Logistic Regression) ==========

Preparing Logistic Regression with correlation_098...
Train accuracy: 0.9560 | Train F1: 0.9558

Preparing Logistic Regression with filtered...
Train accuracy: 0.9595 | Train F1: 0.9593

Preparing Logistic Regression with log...
Train accuracy: 0.9737 | Train F1: 0.9736

========== SVC (models\stage2\unscaled\SVC) ==========

Preparing SVC with correlation_098...
Train accuracy: 0.9410 | Train F1: 0.9409

Preparing SVC with filtered...
Train accuracy: 0.9372 | Train F1: 0.9370

Preparing SVC with log...
Train accuracy: 0.6914 | Train F1: 0.6409

========== MLP (models\stage2\unscaled\MLP) ==========

Preparing MLP with correlation_098...
Train accuracy: 0.9792 | Train F1: 0.9791

Preparing MLP with filtered...
Train accuracy: 0.9724 | Train F1: 0.9723

Preparing MLP with log...
Train accuracy: 0.9820 | Train F1: 0.9820

========== Decision Tree (models\stage2\un

## Evaluation

In [8]:
from src.evaluation import evaluate_models, save_results
results = evaluate_models(saved)
save_results(results, "results/", "results_stage_II.json")

Evaluating Logistic Regression on correlation_098...
Evaluating Logistic Regression on filtered...
Evaluating Logistic Regression on log...
Evaluating SVC on correlation_098...
Evaluating SVC on filtered...
Evaluating SVC on log...
Evaluating MLP on correlation_098...
Evaluating MLP on filtered...
Evaluating MLP on log...
Evaluating Decision Tree on correlation_098...
Evaluating Decision Tree on filtered...
Evaluating Decision Tree on log...
Results saved to: C:\Users\Monon\university\4th-year-bachelor\tfg\results\results_stage_II.json


In [9]:
results

,Model,Feature Set,Features Count,Train Time,Train Accuracy,Train F1,BENIGN,DDoS,FP,FN,TP,TN,FP Rate,Accuracy,F1,Features Names
0,Logistic Regression,correlation_098,43,20.229,0.955980,0.955802,9760,12801,769,215,12586,8991,0.078791,0.956385,0.956210,"[Flow Duration, Total Fwd Packets, Total Backw..."
1,Logistic Regression,filtered,59,21.358,0.959476,0.959270,9760,12801,811,116,12685,8949,0.083094,0.958911,0.958696,"[Flow Duration, Total Fwd Packets, Total Backw..."
2,Logistic Regression,log,59,20.611,0.973700,0.973592,9760,12801,586,12,12789,9174,0.060041,0.973494,0.973383,"[Flow Duration, Total Fwd Packets, Total Backw..."
3,SVC,correlation_098,43,2898.864,0.941017,0.940898,9760,12801,787,543,12258,8973,0.080635,0.941049,0.940954,"[Flow Duration, Total Fwd Packets, Total Backw..."
4,SVC,filtered,59,4785.073,0.937171,0.937027,9760,12801,849,563,12238,8911,0.086988,0.937414,0.937295,"[Flow Duration, Total Fwd Packets, Total Backw..."
5,SVC,log,59,2243.156,0.691384,0.640932,9760,12801,6934,73,12728,2826,0.710451,0.689420,0.638075,"[Flow Duration, Total Fwd Packets, Total Backw..."
6,MLP,correlation_098,43,40.355,0.979162,0.979099,9760,12801,437,18,12783,9323,0.044775,0.979832,0.979774,"[Flow Duration, Total Fwd Packets, Total Backw..."
7,MLP,filtered,59,38.152,0.972375,0.972256,9760,12801,576,17,12784,9184,0.059016,0.973716,0.973609,"[Flow Duration, Total Fwd Packets, Total Backw..."
8,MLP,log,59,76.496,0.981979,0.981953,9760,12801,300,84,12717,9460,0.030738,0.982979,0.982955,"[Flow Duration, Total Fwd Packets, Total Backw..."
9,Decision Tree,correlation_098,43,2.147,0.999941,0.999941,9760,12801,10,9,12792,9750,0.001025,0.999158,0.999158,"[Flow Duration, Total Fwd Packets, Total Backw..."


## Feature Scaling Experiment

Evaluate the effect of standarization for the scale-sensitive models. The experiment was conducted on the 43-feature correlation-reduced representation to assess whether scaling could further improve the selected compact feature representation.

---

In [10]:
DATASETS = {
    "correlation_098_scaled": "data/processed/stage2/full-225k-correlation_098.parquet",
}

In [11]:
from src.models import get_baseline_models 
models_scaled = get_baseline_models()

In [12]:
from src.preprocessing import load_dataset, split_features_target, encode_target

X_SETS = {}
Y_SETS = {}

for feature_set_name, path in DATASETS.items():

    df = load_dataset(path)
    X, y = split_features_target(df,target_column="Label")
    y = encode_target(y)

    X_SETS[feature_set_name] = X
    Y_SETS[feature_set_name] = y

In [13]:
for name, X in X_SETS.items():
    print(f"{name}: {X.shape}")

correlation_098_scaled: (225603, 43)


In [14]:
from src.train import train
saved_scaled = train(
    X_SETS,
    Y_SETS,
    models_scaled,
    dir_name="models/stage2/scaled",
    scaler="standard",
    test_size=0.1
)

Training models on every feature set

========== Logistic Regression (models\stage2\scaled\Logistic Regression) ==========

Preparing Logistic Regression with correlation_098_scaled...
Train accuracy: 0.9982 | Train F1: 0.9982

========== SVC (models\stage2\scaled\SVC) ==========

Preparing SVC with correlation_098_scaled...
Train accuracy: 0.9988 | Train F1: 0.9988

========== MLP (models\stage2\scaled\MLP) ==========

Preparing MLP with correlation_098_scaled...
Train accuracy: 0.9993 | Train F1: 0.9993


In [15]:
from src.evaluation import evaluate_models, save_results
results_scaled = evaluate_models(saved_scaled)
save_results(results_scaled, "results/", "results_stage_II_scaled.json")

Evaluating Logistic Regression on correlation_098_scaled...
Evaluating SVC on correlation_098_scaled...
Evaluating MLP on correlation_098_scaled...
Results saved to: C:\Users\Monon\university\4th-year-bachelor\tfg\results\results_stage_II_scaled.json


In [16]:
results_scaled

,Model,Feature Set,Features Count,Train Time,Train Accuracy,Train F1,BENIGN,DDoS,FP,FN,TP,TN,FP Rate,Accuracy,F1,Features Names
0,Logistic Regression,correlation_098_scaled,43,3.774,0.998168,0.998168,9760,12801,24,16,12785,9736,0.002459,0.998227,0.998227,"[Flow Duration, Total Fwd Packets, Total Backw..."
1,SVC,correlation_098_scaled,43,40.753,0.998754,0.998754,9760,12801,17,15,12786,9743,0.001742,0.998582,0.998582,"[Flow Duration, Total Fwd Packets, Total Backw..."
2,MLP,correlation_098_scaled,43,25.339,0.999325,0.999325,9760,12801,7,13,12788,9753,0.000717,0.999114,0.999114,"[Flow Duration, Total Fwd Packets, Total Backw..."
